# 🎬 clipsmith — Crea clips de tus VODs de Twitch

Convierte tus VODs de Twitch en clips verticales listos para **TikTok, Instagram Reels y YouTube Shorts**, usando inteligencia artificial de forma completamente gratuita.

---

**¿Qué hace este cuaderno?**
1. Instala todo lo necesario automáticamente
2. Instala la IA local — sin costos ni cuentas externas
3. Descarga tu VOD de Twitch
4. Transcribe el audio con IA (~8–12 min para un VOD de 2 horas)
5. Selecciona los mejores momentos automáticamente
6. Genera los clips y los guarda en tu Google Drive

---

**Antes de empezar — haz esto una sola vez:**

Cambia el tipo de ejecución a GPU: ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** y guarda.

> ⚠️ No cierres esta pestaña mientras se ejecutan los pasos. Si el entorno se desconecta, vuelve al Paso 1.

In [ ]:
# @title Paso 1: Instalar herramientas ▶ Ejecuta esto una vez al inicio
import subprocess

print("⏳ Instalando herramientas... esto puede tardar 1-2 minutos.")
get_ipython().system('pip install -q "clipsmith-ai[ollama]"')
get_ipython().system("pip install -q faster-whisper nvidia-cublas-cu11 nvidia-cudnn-cu11")

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
)
gpu = result.stdout.strip()
if gpu:
    print(f"✅ GPU lista: {gpu} — la transcripción será rápida")
else:
    print("⚠️  GPU no detectada. Ve a Entorno de ejecución → Cambiar tipo → T4 GPU y reinicia.")

print("✅ Herramientas instaladas.")

In [ ]:
# @title Paso 2: Credenciales de Twitch (opcional)
# Si no tienes credenciales de Twitch, el cuaderno funciona igual.
# Con credenciales, los clips se seleccionan con más contexto del chat.
# Para obtenerlas: https://dev.twitch.tv/console
import os

try:
    from google.colab import userdata

    os.environ["TWITCH_CLIENT_ID"] = userdata.get("TWITCH_CLIENT_ID") or ""
    os.environ["TWITCH_CLIENT_SECRET"] = userdata.get("TWITCH_CLIENT_SECRET") or ""
    if os.environ["TWITCH_CLIENT_ID"]:
        print("✅ Credenciales de Twitch cargadas.")
    else:
        print("ℹ️  Sin credenciales de Twitch — funcionará igual, con menos señales del chat.")
except Exception:
    print("ℹ️  Sin credenciales de Twitch — funcionará igual, con menos señales del chat.")

In [ ]:
# @title Paso 3: Instalar la IA local ▶ Puede tardar 3-5 minutos
import subprocess
import threading
import time

MODELO_IA = "llama3.1:8b"  # No cambies esto

print("⏳ Instalando servidor de IA local...")
get_ipython().system("apt-get install -y -q pciutils zstd")
get_ipython().system("curl -fsSL https://ollama.com/install.sh | sh")


def _start_ollama():
    subprocess.Popen(["ollama", "serve"])


threading.Thread(target=_start_ollama).start()
time.sleep(5)

print(f"⏳ Descargando modelo de IA ({MODELO_IA}, ~4.7 GB)... unos minutos más.")
get_ipython().system(f"ollama pull {MODELO_IA}")
print("✅ IA lista.")

In [ ]:
# @title ✏️ Paso 4: Configura tu VOD — ¡EDITA AQUÍ ANTES DE CONTINUAR!
from pathlib import Path

# ════════════════════════════════════════════════════════
#   RELLENA ESTOS DATOS — luego ejecuta la celda
# ════════════════════════════════════════════════════════

VOD_ID = "2763505810"  # ID del VOD (los números al final de la URL de Twitch)
IDIOMA = "es"  # Idioma del stream: "es" español · "en" inglés
STREAMER = "chuyelwero"  # Tu nombre de streamer
JUEGO = "Hollow Knight"  # Nombre del juego
FECHA = "2026-05-03"  # Fecha del stream (AAAA-MM-DD)
MAX_CLIPS = 20  # Número máximo de momentos a analizar

# ════════════════════════════════════════════════════════
#   No edites debajo de esta línea
# ════════════════════════════════════════════════════════
config_yaml = f"""\
work_dir: /content/work
out_dir:  /content/out

llm:
  provider: ollama
  model_ollama: {MODELO_IA}

transcribe:
  model: medium
  compute_type: float16
  language: {IDIOMA}
  chunk_minutes: 10
  max_workers: 2

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none

caption:
  enabled: false
"""

Path("/content/work").mkdir(parents=True, exist_ok=True)
Path("/content/out").mkdir(parents=True, exist_ok=True)
Path("config.yaml").write_text(config_yaml)

cached = Path(f"/content/work/{VOD_ID}/transcript.json")
if cached.exists():
    cached.unlink()

print("✅ Configuración guardada:")
print(f"   VOD:      {VOD_ID}")
print(f"   Streamer: {STREAMER}")
print(f"   Juego:    {JUEGO}")
print(f"   Fecha:    {FECHA}")
print(f"   Idioma:   {IDIOMA}")

In [ ]:
# @title 🚀 Paso 5: ¡Crear clips! ▶ Tarda ~8-12 min para un VOD de 2 horas
import sys

print("⏳ Iniciando el proceso completo...")
print("   Descarga → Transcripción → Selección de momentos → Corte de clips")
print("   El progreso aparece aquí abajo. No cierres esta pestaña.\n")

cmd = " ".join(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "run-vod",
        VOD_ID,
        "--config",
        "config.yaml",
        "--max-candidates",
        str(MAX_CLIPS),
        "-v",
    ]
)
ret = get_ipython().system(cmd)
if ret:
    raise RuntimeError(f"El proceso falló (código {ret}). Revisa los mensajes de arriba.")
print("\n✅ ¡Clips creados correctamente! Continúa al Paso 6.")

In [ ]:
# @title 👀 Paso 6: Ver tus clips
from IPython.display import Video, display
from pathlib import Path

clips = sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4"))

if not clips:
    print("❌ No se encontraron clips. ¿Completaste el Paso 5 sin errores?")
else:
    print(f"✅ Se generaron {len(clips)} clip(s):\n")
    for mp4 in clips:
        print(f"  📹 {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))

In [ ]:
# @title 💾 Paso 7: Guardar clips en Google Drive
import shutil
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

clips = sorted((Path("/content/out") / VOD_ID).glob("clip_*.mp4"))

if not clips:
    print("❌ No se encontraron clips. ¿Completaste el Paso 5?")
else:
    print("Selecciona los clips que quieres guardar (todos marcados por defecto):\n")
    checkboxes = [
        widgets.Checkbox(value=True, description=mp4.name, layout=widgets.Layout(width="600px"))
        for mp4 in clips
    ]
    display(*checkboxes)

    def guardar_clips(btn):
        from google.colab import drive

        drive.mount("/content/drive")
        dest = Path(f"/content/drive/MyDrive/{STREAMER}/{JUEGO}/{FECHA}")
        dest.mkdir(parents=True, exist_ok=True)
        selected = [clips[i] for i, cb in enumerate(checkboxes) if cb.value]
        if not selected:
            print("⚠️  No seleccionaste ningún clip.")
            return
        for mp4 in selected:
            shutil.copy2(mp4, dest / mp4.name)
            print(f"✅ Guardado: {mp4.name}")
        print(f"\n🎉 ¡Listo! {len(selected)} clip(s) guardados en:")
        print(f"   Mi unidad → {STREAMER} → {JUEGO} → {FECHA}")

    btn = widgets.Button(
        description="Guardar en Drive", button_style="success", icon="cloud-upload"
    )
    btn.on_click(guardar_clips)
    print()
    display(btn)